Propósito: Filtra las empresas del SPY500 para traer los papeles en sobreventa, llamado de ahora en más "gangas".Usa parametros de los indicadores RSI, MACD y Koncorde en la temporalidad de 60min.

Este filtro busca capturar movimientos que duran entre 2 a 5 días.

⚠ Solo funciona en Google Colab!

In [ ]:
#Scrapea Wikipedia para acceder a los nombres de los tickers del S&P500.

import pandas as pd
import requests
import io

#Define la URL y un 'User-Agent' para evitar el bloqueo (HTTPError).
url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
header = {
  "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/50.0.2661.75 Safari/537.36",
  "X-Requested-With": "XMLHttpRequest"
}

#Realiza la petición con el header de seguridad
r = requests.get(url, headers=header)

#Lee la tabla desde el texto de la respuesta, usando io.StringIO para forzar la interpretación como string HTML
tables = pd.read_html(io.StringIO(r.text))
df_spy = tables[0]

#Extrae la lista de Tickers
tickers_a_analizar = df_spy['Symbol'].tolist()

print(f"Se han cargado {len(tickers_a_analizar)} activos del SPY.")
print("tickers_a_analizar:", tickers_a_analizar[:600])

Se han cargado 503 activos del SPY.
tickers_a_analizar: ['MMM', 'AOS', 'ABT', 'ABBV', 'ACN', 'ADBE', 'AMD', 'AES', 'AFL', 'A', 'APD', 'ABNB', 'AKAM', 'ALB', 'ARE', 'ALGN', 'ALLE', 'LNT', 'ALL', 'GOOGL', 'GOOG', 'MO', 'AMZN', 'AMCR', 'AEE', 'AEP', 'AXP', 'AIG', 'AMT', 'AWK', 'AMP', 'AME', 'AMGN', 'APH', 'ADI', 'AON', 'APA', 'APO', 'AAPL', 'AMAT', 'APP', 'APTV', 'ACGL', 'ADM', 'ARES', 'ANET', 'AJG', 'AIZ', 'T', 'ATO', 'ADSK', 'ADP', 'AZO', 'AVB', 'AVY', 'AXON', 'BKR', 'BALL', 'BAC', 'BAX', 'BDX', 'BRK.B', 'BBY', 'TECH', 'BIIB', 'BLK', 'BX', 'XYZ', 'BK', 'BA', 'BKNG', 'BSX', 'BMY', 'AVGO', 'BR', 'BRO', 'BF.B', 'BLDR', 'BG', 'BXP', 'CHRW', 'CDNS', 'CPT', 'CPB', 'COF', 'CAH', 'CCL', 'CARR', 'CVNA', 'CAT', 'CBOE', 'CBRE', 'CDW', 'COR', 'CNC', 'CNP', 'CF', 'CRL', 'SCHW', 'CHTR', 'CVX', 'CMG', 'CB', 'CHD', 'CI', 'CINF', 'CTAS', 'CSCO', 'C', 'CFG', 'CLX', 'CME', 'CMS', 'KO', 'CTSH', 'COIN', 'CL', 'CMCSA', 'FIX', 'CAG', 'COP', 'ED', 'STZ', 'CEG', 'COO', 'CPRT', 'GLW', 'CPAY', 'CTVA', 'CSGP', 'CO

In [ ]:
#Importar Pandas TA para utilizar los indicadores de análisis técnico. Puede que tengas que ejecutar esta celda dos veces.
!pip install pandas-ta -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 240.3/240.3 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 72.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 133.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 MB 17.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.0 which is incompatible.
db-dtypes 1.5.0 requires pandas<3.0.0,>=1.5.3, but you have pandas 3.0.0 which is incompatible.
bqplot 0.12.45 requires pandas<3.0.0,>=1.0.0, but you have pandas 3.0.0 which is 

In [ ]:
import yfinance as yf
import pandas_ta as ta
import pandas as pd

def calcular_koncorde(df):

    rango_min = df['Low'].rolling(window=14).min()
    rango_max = df['High'].rolling(window=14).max()

    mano_debil = ((df['Close'] - rango_min) / (rango_max - rango_min) * 100) - 50
    return mano_debil.iloc[-1], mano_debil.iloc[-2]

# Limpiar tickets (Si no existen, usamos una lista pequeña de prueba)
try:
    tickers_limpios = [t.replace('.', '-') for t in tickers_a_analizar]
except NameError:
    print("La lista 'tickers_a_analizar' no existía. Usando lista de prueba (FAANG)...")
    tickers_limpios = ['AAPL', 'AMZN', 'GOOGL', 'MSFT', 'TSLA', 'META', 'NVDA']

# DESCARGA Y ANÁLISIS.
print(f"Descargando datos para {len(tickers_limpios)} activos...")
data_60m = yf.download(tickers_limpios, period="3mo", interval="60m", group_by='ticker', progress=False)

empresas_60min = []

for ticker in tickers_limpios:
    try:
        df = data_60m[ticker].dropna()
        if len(df) < 50: continue

        # Cálculos Técnicos
        df.ta.rsi(length=10, append=True)
        df['RSI_MA'] = ta.sma(df['RSI_10'], length=14)
        df.ta.macd(fast=12, slow=26, signal=9, append=True)

        # Variables para validación (nombres generados por pandas_ta)
        rsi_act = df['RSI_10'].iloc[-1]
        rsi_ant = df['RSI_10'].iloc[-2]
        rsi_ma = df['RSI_MA'].iloc[-1]
        macd_act = df['MACD_12_26_9'].iloc[-1]
        sig_act = df['MACDs_12_26_9'].iloc[-1]
        kon_act, kon_ant = calcular_koncorde(df)

        precio_act = df['Close'].iloc[-1]
        precio_ant = df['Close'].iloc[-2]

        # Lógica de Filtrado 60min
        if rsi_act < 35 and macd_act < sig_act and kon_act < 0:
            empresas_60min.append({
                'Ticker': ticker,
                'Prec_Act': round(precio_act, 2),
                'Prec_Ant': round(precio_ant, 2),
                'RSI_Act': round(rsi_act, 1),
                'RSI_Ant': round(rsi_ant, 1),
                'RSI_MA': round(rsi_ma, 1),
                'Konk_Act': round(kon_act, 1),
                'Konk_Ant': round(kon_ant, 1)
            })
    except Exception as e:
      print(f"Error en {ticker}: {e}")

print("\n" + "="*95)
if empresas_60min:
    df_final = pd.DataFrame(empresas_60min)
    print(df_final.sort_values(by='Konk_Act').to_string(index=False))
else:
    print("Sin señales detectadas.")

Descargando datos para 503 activos...


/tmp/ipython-input-1134194847.py:22: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data_60m = yf.download(tickers_limpios, period="3mo", interval="60m", group_by='ticker', progress=False)



Ticker  Prec_Act  Prec_Ant  RSI_Act  RSI_Ant  RSI_MA  Konk_Act  Konk_Ant
  ARES    149.68    149.99     12.7     13.2    16.3     -49.1     -49.7
  ULTA    660.80    661.79     28.8     29.7    44.3     -48.9     -46.1
  STLD    173.31    174.13     26.8     29.4    48.7     -48.1     -49.8
   PSA    278.66    279.77     26.8     30.3    30.6     -47.8     -36.7
   TPR    126.13    126.64     27.7     30.6    36.1     -47.2     -35.9
   MPC    172.50    172.90     33.9     36.0    40.6     -47.2     -43.0
   EXR    140.01    140.48     25.7     28.7    27.2     -47.0     -36.8
   URI    911.08    915.85     29.5     34.7    34.5     -46.9     -39.1
   SYY     75.65     76.09     32.3     40.8    38.3     -46.7     -23.3
  SMCI     30.78     30.75     33.4     32.7    42.2     -46.0     -48.9
  INTC     42.49     42.38     21.8     20.3    21.7     -45.9     -49.1
   UPS    106.93    107.20     30.2     33.9    38.2     -44.7     -46.5
  QCOM    154.53    154.77     33.9     35.9    37

In [ ]:


# DESCARGA Y ANÁLISIS SEMANAL
print(f"Descargando datos SEMANALES para {len(tickers_limpios)} activos...")
data_7d = yf.download(tickers_limpios, period="2y", interval="1wk", group_by='ticker', progress=False)

empresas_semanal = []

for ticker in tickers_limpios:
    try:
        df = data_7d[ticker].dropna()

        if len(df) < 50: continue

        # Cálculos Técnicos (La misma lógica, distinta escala temporal)
        df.ta.rsi(length=10, append=True)
        df['RSI_MA'] = ta.sma(df['RSI_10'], length=14)
        df.ta.macd(fast=12, slow=26, signal=9, append=True)

        # Variables de validación
        rsi_act = df['RSI_10'].iloc[-1]
        macd_act = df['MACD_12_26_9'].iloc[-1]
        sig_act = df['MACDs_12_26_9'].iloc[-1]
        kon_act, kon_ant = calcular_koncorde(df)

        precio_act = df['Close'].iloc[-1]

        # LÓGICA DE FILTRADO MACRO
        if rsi_act < 40 and macd_act < sig_act and kon_act < 0:
            empresas_semanal.append({
                'Ticker': ticker,
                'Prec_Seman': round(precio_act, 2),
                'RSI_Sem': round(rsi_act, 1),
                'RSI_MA_S': round(df['RSI_MA'].iloc[-1], 1),
                'Konk_S_Act': round(kon_act, 1),
                'Konk_S_Ant': round(kon_ant, 1)
            })
    except:
        continue

print("\n" + "="*95)


print("GANGAS DETECTADAS EN SEMANAL (VISIÓN MACRO):")
if empresas_semanal:
    df_final = pd.DataFrame(empresas_semanal)
    # Ordenamos por Koncorde para ver dónde el pánico semanal es más extremo
    print(df_final.sort_values(by='Konk_S_Act').to_string(index=False))
else:
    print("Sin señales semanales.")

Descargando datos SEMANALES para 503 activos...


/tmp/ipython-input-1919644792.py:3: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data_7d = yf.download(tickers_limpios, period="2y", interval="1wk", group_by='ticker', progress=False)



GANGAS DETECTADAS EN SEMANAL (VISIÓN MACRO):
Ticker  Prec_Seman  RSI_Sem  RSI_MA_S  Konk_S_Act  Konk_S_Ant
   CEG      285.27     29.7      51.0       -49.3       -47.0
  COIN      213.48     31.3      39.3       -48.4       -49.2
  QCOM      154.52     38.9      54.2       -47.5       -45.0
   BXP       64.77     37.9      46.7       -47.0       -41.6
   HPQ       19.46     24.5      37.3       -47.0       -47.4
   FIS       59.10     28.9      39.9       -46.4       -35.6
  PAYX      106.19     30.9      28.0       -45.2       -49.2
  PODD      271.31     29.4      42.5       -43.8       -43.5
  PYPL       56.60     27.2      34.3       -43.4       -43.3
  PAYC      147.45     27.1      25.6       -42.3       -39.1
  PSKY       11.89     34.1      41.6       -41.7       -45.2
   BBY       66.75     37.7      47.3       -41.0       -39.4
  GDDY      106.02     21.0      25.9       -41.0       -47.6
   CDW      127.40     28.7      35.0       -40.6       -44.5
  SWKS       59.76     3

In [ ]:
# FRACTALES ALINEADOS, CRUCE DE 60 MINUTOS AND SEMANAL.

# Convertimos las listas de diccionarios en DataFrames (si no lo eran ya)
df_60 = pd.DataFrame(empresas_60min)
df_sem = pd.DataFrame(empresas_semanal)
print("Alineando las temporalidades.")
if not df_60.empty and not df_sem.empty:
    # Solo quedarán los activos que estén presentes en AMBOS DataFrames
    gangas_alineadas = pd.merge(df_60, df_sem, on='Ticker')


    if not gangas_alineadas.empty:
        print("\n" + "!"*30)
        print("ALINEACIÓN DE TEMPORALIDAD DETECTADA")
        print("Activos en zona de pánico tanto en 60min como en 7d:")
        # Seleccionamos las columnas más relevantes para la comparativa
        columnas_finales = [
            'Ticker', 'Prec_Act', 'RSI_Act', 'Konk_Act',
            'RSI_Sem', 'Konk_S_Act'
        ]
        print(gangas_alineadas[columnas_finales].to_string(index=False))

        print("\nInterpretación: El activo está en suelo macro y dando señal de giro micro.")
    else:
        print("\nNo hay coincidencias 'AND'.")
        print("Existen gangas individuales, pero no están alineadas en ambos fractales hoy.")
else:
    if df_60.empty:
        print("- El fractal de 60 min no detectó señales.")
    if df_sem.empty:
        print("- El fractal Semanal no detectó señales.")
    print("\nNo se puede realizar la operación AND sin datos en ambos marcos.")

Alineando las temporalidades.

!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
ALINEACIÓN DE TEMPORALIDAD DETECTADA
Activos en zona de pánico tanto en 60min como en 7d:
Ticker  Prec_Act  RSI_Act  Konk_Act  RSI_Sem  Konk_S_Act
   FIS     59.08     28.7     -37.7     28.9       -46.4
   KKR    118.64     27.1     -24.7     39.2       -29.3
  QCOM    154.53     33.9     -43.8     38.9       -47.5
  SMCI     30.78     33.4     -46.0     36.0       -38.4

Interpretación: El activo está en suelo macro y dando señal de giro micro.


In [ ]:
# SEMÁFORO Y REVERSIÓN DE CALIDAD

UMBRAL_PANICO = 35       # Para activos "estirados" al extremo (Cuchillos)
UMBRAL_CALIDAD = 48      # Para captar reversiones inminentes en líderes
EMPRESAS_FILTRADAS = []

print(f"Iniciando escáner sobre {len(tickers_limpios)} activos...")

for ticker in tickers_limpios:
    try:
        # Reutilizamos los datos de la celda de 60 min (Micro Fractal)
        df = data_60m[ticker].dropna()
        if len(df) < 50: continue

        # Cálculos Técnicos (Referenciando el fractal de 1h)
        df.ta.rsi(length=10, append=True)
        df['RSI_MA'] = ta.sma(df['RSI_10'], length=14)

        # Variables de control
        rsi_act = df['RSI_10'].iloc[-1]
        rsi_ant = df['RSI_10'].iloc[-2]
        rsi_ma = df['RSI_MA'].iloc[-1]
        kon_act, kon_ant = calcular_koncorde(df)

        # FILTRO ESTIRADO (Filtro de Entrada)
        # Captura el pánico real O sobreventa relativa en empresas fuertes
        if rsi_act < UMBRAL_CALIDAD and kon_act < 0:

            # LÓGICA DEL SEMÁFORO (El Gatillo)
            # Caso Verde: Cruce de Media al alza (Ayer < MA y Hoy > MA)
            if rsi_act > rsi_ma and rsi_ant <= rsi_ma:
                estado = "🟢 GATILLO (Cruce confirmado)"
            # Caso Amarillo: Frenado de caída (Ayer < Hoy pero aún debajo de MA)
            elif rsi_act > rsi_ant:
                estado = "🟡 AGOTAMIENTO (En observación)"
            # Caso Rojo: Inercia bajista (Sigue cayendo)
            else:
                estado = "🔴 CUCHILLO (Sin señal de giro)"

            # Buscamos si el ticker también dio señal en la lista semanal previa
            es_ganga_macro = any(e['Ticker'] == ticker for e in empresas_semanal)
            etiqueta_macro = "🔥 ALINEADO (Macro+Micro)" if es_ganga_macro else "Solo Micro"

            EMPRESAS_FILTRADAS.append({
                'Ticker': ticker,
                'Estado': estado,
                'RSI': round(rsi_act, 1),
                'RSI_MA': round(rsi_ma, 1),
                'Tipo': "Extremo" if rsi_act < UMBRAL_PANICO else "Calidad",
                'Fractal': etiqueta_macro
            })

    except Exception:
        continue

# PRESENTACIÓN DE RESULTADOS
print("\n" + "="*100)
print(f"REPORTE DEL S&P500 - {len(EMPRESAS_FILTRADAS)} ACTIVOS EN RADAR")
print("="*100)

if EMPRESAS_FILTRADAS:
    df_final = pd.DataFrame(EMPRESAS_FILTRADAS)

    # Ordenamos por prioridad: Gatillos primero, luego Agotamiento, luego Cuchillos
    orden_estado = {"🟢 GATILLO (Cruce confirmado)": 0, "🟡 AGOTAMIENTO (En observación)": 1, "🔴 CUCHILLO CAYENDO (Sin señal de giro)": 2}
    df_final['Sort'] = df_final['Estado'].map(orden_estado)
    df_final = df_final.sort_values(by=['Sort', 'RSI']).drop(columns='Sort')

    print(df_final.to_string(index=False))
else:
    print("El mercado no presenta activos por debajo de RSI 48 con pánico de mano débil.")

Iniciando escáner sobre 503 activos...

REPORTE DE OPORTUNIDADES - 157 ACTIVOS EN RADAR
Ticker                         Estado  RSI  RSI_MA    Tipo                  Fractal
  INTC   🟢 GATILLO (Cruce confirmado) 21.8    21.7 Extremo               Solo Micro
   APO   🟢 GATILLO (Cruce confirmado) 26.8    21.2 Extremo               Solo Micro
   KKR   🟢 GATILLO (Cruce confirmado) 27.1    24.0 Extremo 🔥 ALINEADO (Macro+Micro)
  COIN   🟢 GATILLO (Cruce confirmado) 30.6    29.1 Extremo 🔥 ALINEADO (Macro+Micro)
   PHM   🟢 GATILLO (Cruce confirmado) 36.2    32.7 Calidad               Solo Micro
   HUM   🟢 GATILLO (Cruce confirmado) 37.0    36.3 Calidad               Solo Micro
   COO   🟢 GATILLO (Cruce confirmado) 37.5    35.9 Calidad               Solo Micro
   JBL   🟢 GATILLO (Cruce confirmado) 38.9    36.0 Calidad               Solo Micro
   CMI   🟢 GATILLO (Cruce confirmado) 42.9    37.4 Calidad               Solo Micro
  ODFL   🟢 GATILLO (Cruce confirmado) 43.5    40.6 Calidad              